## **✅ Lession 1: LangChain - Foundational Models**
- Refrence Link: https://docs.langchain.com/oss/python/langchain/agents
- Initialize LLMs and invoking LLMs.

- Create Agents -> Initializing and invoking agent.
- Create agent with system prompt, human message, streaming response and response format using Pydantic Model.
- Stream the response from the agent and print it in real-time as it is generated for token, metadata in agent.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel
from pprint import pprint

In [33]:
load_dotenv()

assert os.getenv("GROQ_API_KEY")
assert os.getenv("LANGSMITH_API_KEY")
assert os.getenv("TAVILY_API_KEY")
# assert os.getenv("OPEN_ROUTER_API_KEY")

### **1. Initialize LLMs**

In [80]:
# Initialize the ChatGroq LLM for general language tasks
llm = ChatGroq(
    model_name="qwen/qwen3-32b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7, #it handles the randomness of the output, higher values (e.g., 0.8) make the output more random, while lower values (e.g., 0.2) make it more focused and deterministic.
    max_tokens=2048, #it limits the maximum number of tokens in the generated response. This helps control the length of the output and can prevent excessively long responses. 
    timeout = 30, #it specifies the maximum time (in seconds) that the model will take to generate a response.If the model takes longer than this time, it will stop and return whatever it has generated so far. This is useful for preventing long waits in case of complex queries or issues with the model.
    max_retries=3, #it defines the maximum number of times to retry a request if it fails due to network issues or other transient errors. This helps improve the robustness of your application by allowing it to recover from temporary problems without crashing.
)

# Initialize the ChatOpenRouter LLM for vision tasks
vision_llm = ChatOpenRouter(
    model="openai/gpt-5-nano",  # confirm active model
    api_key=os.getenv("OPEN_ROUTER_API_KEY"),
    temperature=0.7,
)

### **2. Invoking LLMs**

In [88]:
# Create a chat prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{text}")
])

# Format the prompt with user input
response = vision_llm.invoke([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is Generative ai?."}
])
print(response.content)

Generative AI is a branch of artificial intelligence that focuses on creating new content, such as text, images, music, code, or videos, rather than just recognizing or classifying existing data.

Key ideas:
- It learns from large data sets to understand patterns, structure, and relationships in that data.
- It then generates new samples that resemble what it learned, often by predicting the next word, stroke, or pixel, or by transforming noise into an image.

How it works at a high level:
- Train: The model studies lots of examples to learn how data is typically formed.
- Generate: When asked, it samples from the learned patterns to produce new content that looks like the training data.
- Refine: Some systems can be guided or fine-tuned to fit a style or goal, and may be evaluated or filtered for quality and safety.

Common types/models:
- Autoregressive models (e.g., GPT for text, some code models): predict the next item in a sequence.
- Diffusion models (e.g., Stable Diffusion): sta

### **3. Initializing and invoking Agent**

In [86]:
from langchain.agents import create_agent

# Create an agent with the LLM
agent = create_agent(llm)

response = agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is luna city."),
    HumanMessage(content="Intresting, can you tell me more about luna city?")]})

print(response['messages'][-1].content)

<think>
Okay, the user asked for more information about Luna City after I mentioned it as the capital of the Moon. Let me start by recalling what I said before. I need to expand on that. Luna City is fictional, so I should present it as a speculative concept rather than a real place.

First, I should explain the concept of Luna City in general. Maybe mention that it's a hypothetical city, perhaps inspired by real-world space exploration plans. I should talk about its location—maybe in a crater like Shackleton for solar access. Then, the structure: modular habitats, pressurized domes. Materials could be local regolith and 3D printing. 

Next, infrastructure is important. Transportation systems like maglev trains or subways, energy sources like solar and nuclear. Also, life support systems with recycling water and air. Maybe mention agriculture using hydroponics and algae for oxygen. 

Social aspects: governance, maybe a council or international cooperation. Population could be a mix of 

### **4. Streaming Response**

In [ ]:
# streaming response
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="What is the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is luna city."),
    HumanMessage(content="Intresting, can you tell me more about luna city?")]},
    stream_mode = "messages"):
    
    # token is message chunk with token content and metadata is the additional information about the message chunk, such as its role (e.g., user, assistant) and any other relevant details.
    if token.content:
        print(token.content, end="", flush=True)
        
# pprint(response)
print(response['messages'][-1].content)

### **5. Creating an agents with system prompts and structured output using Pydantic model**

In [ ]:
# Define a Pydantic model for structured output
class CityInfo(BaseModel):
    name: str
    population: int
    description: str

# Create an agent with the LLM, system prompt, and structured output    
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant that provides information about cities.",
    response_format=CityInfo
)

# Asking the agent for information about New York City and streaming the response
question = HumanMessage(content="Can you provide information about New York City?")
for token, metadata in agent.stream(
    {"messages": [question]},
    stream_mode = "messages"):
    if token.content:
        print(token.content, end="", flush=True)   
response = agent.invoke({"messages": [question]})

# pprint the structured response
pprint(response['messages'][-1].content)

In [ ]:
response["structured_response"].description

## **✅ Lession 2: LangChain - Tools**
- Reference Link: https://docs.langchain.com/oss/python/langchain/tools

- React agent: Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.

In [ ]:
from langchain.tools import tool

### **Creating Tool**

In [ ]:
# different ways to define a tool in langchain using the @tool decorator
@tool
def square_root(x: float) -> float:
    """Returns the square root of a number."""
    return x ** 0.5

@tool("square_root")
def tool_square_root(x: float) -> float:
    """Returns the square root of a number."""
    return x ** 0.5

@tool("square_root", description="Returns the square root of a number.")
def tool_square_root2(x: float) -> float:
    return x ** 0.5

### **Adding Tool to agent**

In [ ]:
# Adding tool to agent
agent = create_agent(model=llm,
                     tools=[tool_square_root2],
                     system_prompt="You are a helpful assistant that can calculate the square root of a number using the provided tool."
                     )
question = HumanMessage(content="What is the square root of 16?")
response = agent.invoke({"messages": [question]})
pprint(response['messages'])
print(response["messages"][-1].content)

### **🌎 WebSearch**

#### - **Inquiring about the latest news without implementstion of a web search API..**

In [63]:
# creating an agents with system where the agent is a helpful assistant who having a knowldege about the current events and can provide the latest news updates.
agent = create_agent(model=llm,
                     system_prompt="You are a helpful assistant who having a knowldege about the current events and can provide the latest news updates."
)
question = HumanMessage(content="Who is the current chief minister of New Delhi?")
response = agent.invoke({"messages": [question]})
pprint(response["messages"][-1].content)

('<think>\n'
 'Okay, I need to determine who the current chief minister of New Delhi is. '
 'Let me start by recalling the political structure of India. New Delhi, the '
 'capital, is part of the National Capital Territory (NCT) of Delhi, which has '
 'an elected government. The chief minister is the head of the government '
 'there.\n'
 '\n'
 'As of my last update, the Aam Aadmi Party (AAP) has been in power in Delhi. '
 'The previous chief minister was Arvind Kejriwal, who has been a prominent '
 'figure in the AAP. However, I should check if there have been any recent '
 'changes due to elections or other political developments.\n'
 '\n'
 'The last major election in Delhi was in February 2020, where AAP won. Since '
 "then, there hasn't been a general election in 2023, so the current CM should "
 'still be Arvind Kejriwal. But I need to confirm if there have been any '
 'by-elections or if he has taken a leave of absence. I remember that '
 'sometimes in Indian politics, CMs might t

#### - **Inquiring about the latest news after implementstion of a web search API..**

In [64]:
# implementation of web seaech tool
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

In [66]:
# To install: pip install tavily-python
from tavily import TavilyClient
client = TavilyClient("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")
response = client.search(
    query="Who is the current chief minister of the New Delhi?",
    search_depth="advanced"
)
pprint(response)

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'Who is the current chief minister of the New Delhi?',
 'request_id': '5de5ae39-f119-4dd2-b804-c376fa83629a',
 'response_time': 4.08,
 'results': [{'content': "the chief minister's term is for five years and is "
                         'subject to no term limits. Rekha Gupta is the '
                         'incumbent chief minister since February 2025. [...] '
                         '| Chief Minister of the National Capital Territory '
                         'of Delhi | |\n'
                         ' --- |\n'
                         '| Emblem of the NCT of Delhi | |\n'
                         '| Incumbent Rekha Gupta since 20 February 2025 | |\n'
                         '| Government of Delhi | |\n'
                         '| Type | Head of State Government |\n'
                         '| Member of |  Delhi Legislative Assembly & Council '
                         'of Ministers of Delhi |\n'
         

In [67]:
tavily_client = TavilyClient(os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information about a given query."""
    # web search using tavily client and return the results as a dictionary
    return tavily_client.search(query)
web_search.invoke("Who is chief minister of New Delhi?")

{'query': 'Who is chief minister of New Delhi?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Chief_Minister_of_Delhi',
   'title': 'Chief Minister of Delhi - Wikipedia',
   'content': 'The **chief minister of the National Capital Territory of Delhi** is the head of government of the National Capital Territory of Delhi. | Chief Minister\xa0of the National Capital Territory of Delhi |. | Deputy | Deputy Chief Minister of the National Capital Territory of Delhi |. According to the Constitution of India, the lieutenant governor is the National Capital Territory of Delhi\'s *de jure* head, but *de facto* executive authority rests with its chief minister "Chief Minister (India)"). Since 1952, the National Capital Territory of Delhi has had 7 chief ministers, starting with the Indian National Congress party\'s Chaudhary Brahm Prakash. The longest-serving chief minister, Sheila Dikshit from the Indian National Congress party

In [69]:
agent = create_agent(
    model=llm,
    tools=[web_search],
)
question = HumanMessage(content="Who is the current chief minister of New Delhi?")
response = agent.invoke({"messages": [question]})
pprint(response["messages"])

[HumanMessage(content='Who is the current chief minister of New Delhi?', additional_kwargs={}, response_metadata={}, id='9d47da75-9c49-4b4a-8f8a-992f8818270a'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking who the current chief minister of New Delhi is. Let me think. New Delhi is the capital territory of India, so the position is called the Chief Minister of the National Capital Territory of Delhi. I need to find the latest information because this can change over time.\n\nI remember that the previous Chief Minister was Arvind Kejriwal from the Aam Aadmi Party. He has been in office for a few terms. But I should verify if there\'s been any recent change. Maybe there was an election or a resignation. Since I can\'t browse the internet directly, I should use the web_search function to check the most up-to-date information. Let me search for "current Chief Minister of New Delhi 2023" to get the latest details. That should confirm whether Arvind 

## **✅ Lession 3: LangChain - Short-Term Memory**

- In LangChain agents, they track messages called states, which you can think of as a memory of the agent. The problem is the states aren't being saved from one run to another, so in effect, our agents' memory is being wiped out.
- So, we have to do something that can remember the previous messages, which we can do using checkpointer. This saves a snapshot of the state at the end of each run and then groups them. It would utter runs with the same thread ID. We define the thread ID so that we can group these checkpoints of our state together.
- By default, our agent state tracks a list of messages only, but we can add custom field IDs, etc.

### **No Memory**

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant."
)
question = HumanMessage(content="Hey, My name is Hemant and my favorite food is Indian dal makhani.")
response = agent.invoke({"messages": [question]})
pprint(response["messages"])

In [ ]:
question = HumanMessage(content="What is my name and what is my favorite food?")
response = agent.invoke({"messages": [question]})
pprint(response["messages"])

### **Short-Term Memory**

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver()
)
question = HumanMessage(content="Hey, My name is Hemant and my favorite food is Indian dal makhani.")
config = {"configurable": {"thread_id": "1"}}


response = agent.invoke(
    {"messages": [question]},
    config=config
    )
pprint(response["messages"])

In [ ]:
question = HumanMessage(content="What is my name and what is my favorite food?")

response = agent.invoke(
    {"messages": [question]},
    config=config
    )
pprint(response["messages"])

## **✅ Lession 4: LangChain - Multimodal Messages**
- How to provide image and audio as input to the models?
- Will encode our image and audio files into Base64. This encodes binary, which is base two, to 64 printable characters, or base 64. This enables us to represent binary data and transmit this representation on our text-based communication channels.

In [81]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept=".jpg", multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [82]:
print(uploader.value)

({'name': 'omk-6DhD3VYd9q8-unsplash.jpg', 'type': 'image/jpeg', 'size': 829523, 'content': <memory at 0x1154b0700>, 'last_modified': datetime.datetime(2026, 2, 22, 18, 46, 47, 227000, tzinfo=datetime.timezone.utc)},)


In [83]:
import base64

# Check if any file has been uploaded
if not uploader.value:
    print("Please upload an image and run this cell again!")
else:
    # Take the first file from the tuple
    uploaded_file = uploader.value[0]

    # Get the raw bytes
    img_bytes = uploaded_file["content"].tobytes()

    # Convert to base64
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    print("Uploaded file name:", uploaded_file["name"])
    print("Base64 length:", len(img_b64))

Uploaded file name: omk-6DhD3VYd9q8-unsplash.jpg
Base64 length: 1106032


In [85]:
# Build multimodal question
question = HumanMessage(
    content=[
        {"type": "text", "text": "Describe what you see in this image."},
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
        }
    ]
)

# Invoke model
response = vision_llm.invoke([question])
print("Model output:\n")
pprint(response.content)

Model output:

('- A dark, dense forest at night with tall, slender trees and a muddy, rocky '
 'path. \n'
 '- A silhouette of a deer (stag) with prominent antlers stands in the middle '
 'of the path, slightly facing to the left. \n'
 '- Several large, glowing orange orbs sit on the ground and behind trees, '
 'casting a warm orange light through the scene. \n'
 '- The orange illumination creates reflections and a surreal, otherworldly '
 'atmosphere around the deer.')
